In [ ]:
import json
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import Dataset
from transformers import Trainer, TrainingArguments

In [ ]:
# config
MODEL_NAME = 'openai-community/gpt2'
MAX_LENGTH = 512
OUTPUT_DIR = "./gpt2-instruct"

USER_TOKEN = "<|user|>"
ASSISTANT_TOKEN = "<|assistant|>"

In [ ]:
data = json.load(open('instruction-data.json'))

def format_input(example):
    instruction = example['instruction']
    input_text = example.get('input', '').strip()
    output = example['output']

    if input_text:
        user_text = f"{instruction}\n\n{input_text}"
    else:
        user_text = instruction

    text = f"{USER_TOKEN}{user_text}{ASSISTANT_TOKEN}{output}"
    return {"text": text}

dataset = Dataset.from_list([format_input(example) for example in data])

In [ ]:
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
print(len(tokenizer))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


50257


In [ ]:
tokenizer.add_special_tokens({
    'pad_token': tokenizer.eos_token,
    'additional_special_tokens': [USER_TOKEN, ASSISTANT_TOKEN]
})

3

In [ ]:
example = dataset[0]

In [ ]:
def prepare_input(example):
    text = example['text']
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )
    input_ids = enc['input_ids']
    labels = [-100] * len(input_ids)

    assistant_id = tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
    pad_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

    start = input_ids.index(assistant_id) + 1
    end = input_ids.index(pad_id) + 1

    for i in range(start, end):
        labels[i] = input_ids[i]

    return {
        'input_ids': input_ids,
        'attention_mask': enc['attention_mask'],
        'labels': labels
    }

In [ ]:
tokenized_ds = dataset.map(prepare_input, remove_columns=['text'])

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [ ]:
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50259, 768)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # overwrite_output_dir=True,
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=200,
    lr_scheduler_type='cosine',
    optim='adamw_torch',
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    # tokenizer=tokenizer
)

In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Step,Training Loss
500,1.262519


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=552, training_loss=1.2054778527522432, metrics={'train_runtime': 663.9236, 'train_samples_per_second': 6.627, 'train_steps_per_second': 0.831, 'total_flos': 1149684940800000.0, 'train_loss': 1.2054778527522432, 'epoch': 4.0})

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-instruct/tokenizer_config.json', './gpt2-instruct/tokenizer.json')

In [ ]:
from huggingface_hub import login

login()

In [ ]:
HF_REPO = "Sanjarbek1024/gpt2-instruct"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...j5mskey/model.safetensors:   0%|          |  549kB /  498MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Sanjarbek1024/gpt2-instruct/commit/83364913d4d4d8ac6230f8309977757eca914968', commit_message='Upload tokenizer', commit_description='', oid='83364913d4d4d8ac6230f8309977757eca914968', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sanjarbek1024/gpt2-instruct', endpoint='https://huggingface.co', repo_type='model', repo_id='Sanjarbek1024/gpt2-instruct'), pr_revision=None, pr_num=None)

In [ ]:
readme_text = """
# GPT-2 Instruct Model

This is a fine-tuned GPT-2 model for instruction-following tasks.

## Base Model
- openai-community/gpt2

## Training Details
- Epochs: 4
- Batch size: 2
- Max length: 512
- Optimizer: AdamW

## Dataset
- Custom instruction dataset

## Usage Example

from transformers import pipeline

chat = pipeline(
    "text-generation",
    model="Sanjarbek1024/gpt2-instruct",
    tokenizer="Sanjarbek1024/gpt2-instruct"
)

prompt = "<|user|>\\nWhat is AI?\\n<|assistant|>\\n"

result = chat(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

## Author
- Sanjarbek
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("README.md created successfully!")


README.md created successfully!


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="model"
)

print("README uploaded to Hugging Face!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9767: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


README uploaded to Hugging Face!
